# Audio Preprocessing Pipeline (SQL-backed)

Reproducible, end-to-end **processing** for the speech-emotion dataset (EDA lives in a separate notebook):

1. Fixed random seeds (numpy / random / torch / split = **42**)
2. Connect to the cloud MySQL once (TLS); credentials from `.env`
3. Build a **metadata table** (one row per file) with `duplicate_flagged` / `corrupted_flagged` / source / duration
4. **Filter to the clean working set**: reduce to CREMA-D + RAVDESS, drop the misspelled 'Suprised', exclude corrupted/duplicate files
5. **Trim** every clean file to a fixed length → separate `Emotions_processed/` folder (originals untouched)
6. Extract **fixed-shape features** (log-mel) from the trimmed audio
7. **Stratified train/test split** with the fixed seed → a `split` tag per clip
8. **Push the cleaned + split dataset to MySQL** (`audio_dataset`) — the single source of truth the team queries for modeling
9. Print shapes + class distributions to **verify reproducibility** across the team

> The SQL push is the **last** step and stores only the cleaned dataset. If the DB isn't reachable,
> `mysql_ok` stays False and everything still runs in memory — the pipeline never blocks.

In [ ]:
# === Imports + fixed random seeds (run first, every session) ===
import os, re, hashlib, random
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from tqdm.auto import tqdm

# Load DB credentials from the repo-root .env (gitignored) so the kernel sees MYSQL_* without
# needing shell `export`s. override=True so .env wins over any stale shell exports.
try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(), override=True)
except ImportError:
    pass  # pip install python-dotenv  (falls back to shell env vars if missing)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    _t = "/torch"
except ImportError:
    _t = ""
print(f"seeds set to {SEED}  (numpy/random{_t})")

In [ ]:
# === Config ===
EMO_DIR       = Path("../Emotions")            # raw dataset root: <emotion>/*.wav
PROCESSED_DIR = Path("../Emotions_processed")  # trimmed output (NEVER write into EMO_DIR)
SR            = 16000                           # target sample rate (CREMA native; avoids upsampling)
TARGET_SEC    = 3.0                             # fixed clip length for trimming + features
N_MELS        = 128
HOP           = 512

# Cleaned-dataset scope: reduce to these sources, drop these (misspelled / out-of-scope) emotions.
KEEP_SOURCES = ("CREMA-D", "RAVDESS")
DROP_LABELS  = ("Suprised",)

# NOTE: the rubric's "30 s" example suits long recordings; these clips are ~1-5 s, so 30 s would be
# ~90% silence padding. 3 s covers the large majority with minimal padding -- change TARGET_SEC freely.

def source_of(name):
    if re.match(r"^\d+-\d+-", name):             return "RAVDESS"
    if re.match(r"^\d{4}_", name):               return "CREMA-D"
    if re.match(r"^(OAF|YAF|OA)_", name):        return "TESS"
    if re.match(r"^[a-z]{1,2}\d+\.wav$", name):  return "SAVEE"
    return "other"

In [ ]:
# === Database connection (cloud MySQL, e.g. Aiven) — built ONCE, reused by every DB cell below ===
# Credentials come from .env / shell env (see .env.example). Cloud MySQL enforces TLS, so we always
# pass an SSL arg. If the DB is unreachable, mysql_ok stays False and the pipeline falls back to the
# in-memory table so it never blocks.
from sqlalchemy import create_engine, text

USER = os.getenv("MYSQL_USER", "root")
PWD  = os.getenv("MYSQL_PASSWORD", "")
HOST = os.getenv("MYSQL_HOST", "localhost")
PORT = os.getenv("MYSQL_PORT", "3306")
DB   = os.getenv("MYSQL_DB",   "ser")
SSL_CA = os.getenv("MYSQL_SSL_CA", "")           # path to ca.pem for strict CA verification (optional)
CONNECT_ARGS = {"ssl": {"ca": SSL_CA}} if SSL_CA else {"ssl": {"ssl": True}}

def recreate_table(name, ddl):
    """DROP + CREATE a table with an explicit PK (Aiven sets sql_require_primary_key=ON)."""
    with engine.begin() as conn:
        conn.execute(text(f"DROP TABLE IF EXISTS {name}"))
        conn.execute(text(ddl))

mysql_ok = False
try:
    engine = create_engine(f"mysql+pymysql://{USER}:{PWD}@{HOST}:{PORT}/{DB}", connect_args=CONNECT_ARGS)
    with engine.connect() as conn:
        v = conn.execute(text("SELECT VERSION()")).scalar()
    mysql_ok = True
    print(f"OK  connected to {HOST}:{PORT}/{DB}  (MySQL {v})")
except Exception as e:
    print(f"NOT connected ({type(e).__name__}: {e})")
    print("check: .env values set? service Running? port is the 5-digit Aiven port, not 3306?")
    print("-> pipeline will continue with local (in-memory) filtering.")

In [ ]:
# === Build the metadata table (one row per file) ===
# Reads only file headers (soundfile.info) -> fast. Flags corrupted (unreadable / 0 frames) and
# duplicate (identical bytes) files. Nothing is deleted or moved -- flags only.
def file_md5(p, chunk=1 << 20):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for blk in iter(lambda: f.read(chunk), b""):
            h.update(blk)
    return h.hexdigest()

rows = []
for emo_path in sorted(p for p in EMO_DIR.iterdir() if p.is_dir()):
    for wav in sorted(emo_path.glob("*.wav")):
        rec = {"file_path": str(wav), "label": emo_path.name,
               "source": source_of(wav.name), "file_size": wav.stat().st_size}
        try:
            info = sf.info(str(wav))
            rec["duration_sec"] = round(info.frames / info.samplerate, 3)
            rec["sample_rate"]  = int(info.samplerate)
            rec["channels"]     = int(info.channels)
            rec["corrupted_flagged"] = bool(info.frames == 0)
        except Exception:
            rec.update(duration_sec=None, sample_rate=None, channels=None, corrupted_flagged=True)
        rec["md5"] = file_md5(wav) if not rec["corrupted_flagged"] else None
        rows.append(rec)

meta = pd.DataFrame(rows)
# duplicate = same content hash as an earlier file (first occurrence stays unflagged)
meta["duplicate_flagged"] = meta["md5"].duplicated(keep="first") & meta["md5"].notna()
meta = meta.drop(columns=["md5"])              # hash was only needed to compute duplicate_flagged
meta.insert(0, "clip_id", range(1, len(meta) + 1))
print(f"{len(meta)} files | corrupted: {int(meta['corrupted_flagged'].sum())} | "
      f"duplicates: {int(meta['duplicate_flagged'].sum())}")
meta.head()

In [ ]:
# === Filter to the CLEAN working set (in memory) ===
# Cleaned dataset = not corrupted, not duplicate, reduced to KEEP_SOURCES, with DROP_LABELS removed.
# The full `meta` table (every file) was used for EDA above; from here on we work on `clean` only.
clean = meta[(~meta["corrupted_flagged"])
             & (~meta["duplicate_flagged"])
             & (meta["source"].isin(KEEP_SOURCES))
             & (~meta["label"].isin(DROP_LABELS))].reset_index(drop=True)
print(f"clean files: {len(clean)} / {len(meta)} total  "
      f"({', '.join(KEEP_SOURCES)}, no {'/'.join(DROP_LABELS)})")
print(clean["label"].value_counts())

In [ ]:
# === Trim every clean file to TARGET_SEC -> PROCESSED_DIR (originals untouched) ===
TARGET_LEN = int(TARGET_SEC * SR)

def trim_file(src_path, label):
    out_dir = PROCESSED_DIR / label
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / Path(src_path).name
    if out_path.exists():
        return str(out_path)                       # idempotent: skip already-processed
    y, _ = librosa.load(src_path, sr=SR, mono=True)
    y = y[:TARGET_LEN] if len(y) >= TARGET_LEN else np.pad(y, (0, TARGET_LEN - len(y)))
    sf.write(str(out_path), y, SR)
    return str(out_path)

clean = clean.copy()
clean["processed_path"] = [trim_file(p, l) for p, l in
                           tqdm(list(zip(clean["file_path"], clean["label"])), desc="trim")]
print("trimmed audio ->", PROCESSED_DIR.resolve())

In [ ]:
# === Extract fixed-shape features (log-mel) from the trimmed files ===
# Every clip is now exactly TARGET_SEC long, so all feature arrays share the same shape.
def logmel(path):
    y, _ = librosa.load(path, sr=SR, mono=True)
    m = librosa.feature.melspectrogram(y=y, sr=SR, n_mels=N_MELS, hop_length=HOP)
    return librosa.power_to_db(m, ref=np.max).astype("float32")

X = np.stack([logmel(p) for p in tqdm(clean["processed_path"], desc="features")])
y = clean["label"].to_numpy()
print("X:", X.shape, " y:", y.shape)

In [ ]:
# === Stratified train/test split (fixed seed) ===
# Split at the CLIP level so every clip gets a 'train'/'test' tag stored in the dataset table.
# Same seed + stratify as a plain train_test_split(X, y), so the partition is identical.
from sklearn.model_selection import train_test_split

idx = np.arange(len(clean))
train_idx, test_idx = train_test_split(
    idx, test_size=0.2, random_state=SEED, stratify=clean["label"])

clean = clean.copy()
clean["split"] = "train"
clean.iloc[test_idx, clean.columns.get_loc("split")] = "test"

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
print(f"split: train={len(train_idx)}, test={len(test_idx)}")

In [ ]:
# === Push the CLEANED dataset to MySQL (after all processing) ===
# Single source-of-truth table: one row per clean clip, with its processed path and train/test
# split. No flag columns (every row here is already clean). Built with a PK up front (Aiven
# requires it), then rows appended. The team reads modeling data straight from this table.
DATASET_TABLE = "audio_dataset"
DATASET_DDL = f"""
CREATE TABLE {DATASET_TABLE} (
    clip_id        INTEGER      NOT NULL PRIMARY KEY,
    file_path      VARCHAR(512),
    processed_path VARCHAR(512),
    label          VARCHAR(32),
    source         VARCHAR(16),
    duration_sec   FLOAT,
    sample_rate    INTEGER,
    channels       INTEGER,
    split          VARCHAR(8),
    INDEX idx_{DATASET_TABLE}_label (label),
    INDEX idx_{DATASET_TABLE}_split (split)
)
"""
COLS = ["clip_id", "file_path", "processed_path", "label", "source",
        "duration_sec", "sample_rate", "channels", "split"]

if mysql_ok:
    recreate_table(DATASET_TABLE, DATASET_DDL)
    clean[COLS].to_sql(DATASET_TABLE, engine, if_exists="append", index=False)
    with engine.connect() as conn:
        n = conn.execute(text(f"SELECT COUNT(*) FROM {DATASET_TABLE}")).scalar()
    print(f"wrote {n} rows to MySQL `{DB}.{DATASET_TABLE}`  (cleaned + split dataset)")
else:
    print("MySQL not connected -> push skipped; cleaned dataset lives in the `clean` DataFrame")

In [ ]:
# === Reproducibility check: shapes + class distributions (everyone's numbers should match) ===
print("SEED:", SEED)
print("X_train:", X_train.shape, " X_test:", X_test.shape)
print("\ntrain class distribution:"); print(pd.Series(y_train).value_counts().sort_index())
print("\ntest class distribution:");  print(pd.Series(y_test).value_counts().sort_index())

## Reproducibility & sharing

- This notebook **is** the shared preprocessing script — commit it so the team runs the same file:
  ```
  git add Notebooks/sqltransfer.ipynb && git commit -m "SQL-backed audio preprocessing pipeline" && git push
  ```
- `SEED = 42` is applied to numpy / random / torch / `train_test_split` → identical splits for everyone.
- The MySQL **`audio_dataset`** table is the single source of truth: cleaned clips (CREMA-D + RAVDESS,
  no 'Suprised'), each with its `processed_path` and `split`. Teammates load modeling data directly, e.g.
  ```sql
  SELECT clip_id, processed_path, label FROM audio_dataset WHERE split = 'train';
  ```
- Credentials live in a local **`.env`** (gitignored); copy `.env.example` and fill in your own. Never commit secrets.
- Originals in `Emotions/` are never modified; trimmed audio is written to `Emotions_processed/`.